In [2]:
import geopandas as gpd

villages = gpd.read_file('urban_villages_filtered.shp')

In [3]:
beijing = gpd.read_file('Beijing/北京市全部_Polygon.shp')

In [4]:
city = beijing.to_crs(3857)

In [5]:
city

,NAME,geometry
0,东城区,"POLYGON ((12955520.122 4860061.797, 12955722.1..."
1,西城区,"POLYGON ((12954734.280 4847033.263, 12953988.0..."
2,朝阳区,"POLYGON ((12961978.203 4848115.404, 12961981.0..."
3,朝阳区,"POLYGON ((12979616.703 4873487.698, 12979254.3..."
4,丰台区,"POLYGON ((12930965.636 4849598.921, 12931171.5..."
5,石景山区,"POLYGON ((12941231.831 4850760.820, 12941185.8..."
6,海淀区,"POLYGON ((12948647.729 4850766.228, 12948600.4..."
7,门头沟区,"POLYGON ((12857780.850 4875289.751, 12858096.7..."
8,房山区,"POLYGON ((12937012.199 4804768.985, 12936956.7..."
9,通州区,"POLYGON ((12993155.499 4811291.568, 12993115.8..."


In [6]:
villages

,Area,ID,assign_row,assign_col,geometry
0,0.238053,48,33.0,39.0,"POLYGON ((12973408.791 4848757.915, 12973430.7..."
1,0.005151,243,31.0,33.0,"POLYGON ((12965239.832 4851295.885, 12965274.9..."
2,0.008934,406,27.0,27.0,"POLYGON ((12957516.525 4856968.773, 12957511.1..."
3,0.018067,892,34.0,27.0,"POLYGON ((12957066.472 4847557.453, 12957005.8..."
4,0.042724,2238,40.0,36.0,"POLYGON ((12968721.098 4840411.222, 12968664.8..."
...,...,...,...,...,...
1454,0.048165,15554,31.0,35.0,"POLYGON ((12967471.331 4851353.774, 12967107.7..."
1455,0.004903,15558,21.0,32.0,"POLYGON ((12963156.515 4864347.589, 12963156.9..."
1456,0.038213,15559,40.0,36.0,"POLYGON ((12968426.645 4840788.143, 12968558.8..."
1457,0.028219,15560,31.0,35.0,"POLYGON ((12967589.790 4851511.235, 12967597.8..."


In [7]:
district_names = []

# Iterate over each village
for i, village in villages.iterrows():
    max_area = 0
    best_match = None
    
    # Iterate over each city district
    for j, district in city.iterrows():
        intersection = village.geometry.intersection(district.geometry)
        area = intersection.area
        if area > max_area:
            max_area = area
            best_match = district['NAME']  # Replace 'name' if your district name column is different
    
    district_names.append(best_match)

# Add result to the villages GeoDataFrame
villages['district_name'] = district_names

In [8]:
villages['district_name'].value_counts()

district_name
朝阳区     312
丰台区     302
海淀区     261
昌平区     178
通州区      98
顺义区      93
大兴区      81
房山区      79
石景山区     24
门头沟区     24
东城区       3
怀柔区       2
西城区       2
Name: count, dtype: int64

In [22]:
villages[villages['ID'] == 9318]

,Area,ID,assign_row,assign_col,geometry,district_name
31,0.009604,9318,40.0,10.0,"POLYGON ((12935095.418 4839948.017, 12935096.8...",丰台区


In [10]:
district_village_dict = {
    district: group.reset_index(drop=True)
    for district, group in villages.groupby('district_name')
}

In [13]:
cp = district_village_dict['昌平区']
cy = district_village_dict['朝阳区']

In [19]:
cp['ID']

0       9861
1      12696
2      14036
3      14037
4      14040
       ...  
173    15536
174    15537
175    15538
176    15539
177    15540
Name: ID, Length: 178, dtype: int64

In [20]:
cy[cy['ID'] == 9318.0]

,Area,ID,assign_row,assign_col,geometry,district_name


In [45]:
import pandas as pd
# List of district names you want to merge
target_districts = ['石景山区', '门头沟区', '东城区', '怀柔区', '西城区']

# Filter and collect the GeoDataFrames
dfs_to_merge = [district_village_dict.pop(name) for name in target_districts if name in district_village_dict]

# Concatenate into a single GeoDataFrame
merged_gdf = gpd.GeoDataFrame(pd.concat(dfs_to_merge, ignore_index=True), crs=dfs_to_merge[0].crs)

district_village_dict['remaining'] = merged_gdf

In [48]:
import os

# Make sure the output base directory exists (optional, if you want a common parent folder)
base_dir = '.'  # or set to your preferred output path

for district, gdf in district_village_dict.items():
    # Sanitize folder name to be safe for filesystem
    folder_name = os.path.join(base_dir, district.replace(" ", "_").replace("/", "_"))
    os.makedirs(folder_name, exist_ok=True)

    # Drop the 'district_name' column
    gdf_to_save = gdf.drop(columns='district_name', errors='ignore')

    # Define the output shapefile path
    shp_path = os.path.join(folder_name, 'villages.shp')

    # Save as shapefile
    gdf_to_save.to_file(shp_path)
    # break


In [47]:
district_village_dict['剩余五区'] = district_village_dict.pop('remaining')

In [54]:
s = 0
ss = 0
for district, gdf in district_village_dict.items():
    num = (len(gdf)+ 25)//50
    ss += len(gdf)
    s += num
    print(f"{district} {num}")

丰台区 6
大兴区 2
房山区 2
昌平区 4
朝阳区 6
海淀区 5
通州区 2
顺义区 2
剩余五区 1
30
1459


In [24]:
cp = gpd.read_file('昌平区/villages.shp')

In [30]:
cp[cp['ID']==14308]

,Area,ID,assign_row,assign_col,geometry
30,0.127302,14308,11.0,28.0,"POLYGON ((12959092.981 4878406.402, 12959092.9..."


In [29]:
cp.sort_values(by='ID')

,Area,ID,assign_row,assign_col,geometry
0,0.031864,9861,NaN,NaN,"POLYGON ((12938383.828 4896948.366, 12938383.5..."
1,0.008305,12696,13.0,23.0,"POLYGON ((12951831.680 4875618.168, 12951849.8..."
2,0.052042,14036,3.0,18.0,"POLYGON ((12945346.072 4887815.066, 12945353.7..."
3,0.066322,14037,3.0,19.0,"POLYGON ((12945924.719 4887644.774, 12945876.3..."
4,0.039730,14040,9.0,25.0,"POLYGON ((12954455.051 4880266.113, 12954449.0..."
...,...,...,...,...,...
173,0.087347,15536,4.0,18.0,"POLYGON ((12945721.860 4887035.598, 12945722.8..."
174,0.415260,15537,10.0,28.0,"POLYGON ((12957814.761 4880541.218, 12957833.9..."
175,0.084000,15538,9.0,26.0,"POLYGON ((12955892.970 4881121.321, 12955906.3..."
176,0.043403,15539,4.0,18.0,"POLYGON ((12945676.479 4887196.052, 12945605.0..."
